# 03 — Tier 1: direct LLM baseline (Ollama + Gemma)

The **control group**. Gemma answers our 15 benchmark questions with **no access to the
database**. It cannot run a query, so every number it gives is invented.

We are not trying to make this good. We are measuring how wrong an ungrounded answer is,
so Tier 2 and Tier 3 have something to beat.

Runs locally, so it is free and needs no API key.

**Setup**
```bash
ollama pull gemma3          # or gemma3:4b if you are short on RAM
pip install ollama
```


## 1. Setup


In [ ]:
import json, time
from pathlib import Path

import ollama
import pandas as pd
import yaml
import matplotlib.pyplot as plt

MODEL = 'gemma3'        # check what you have with: ollama list
TOLERANCE = 0.01        # a number counts as correct within 1%

RESULTS = Path('../eval/results')
RESULTS.mkdir(parents=True, exist_ok=True)

questions = yaml.safe_load(Path('../eval/benchmark.yaml').read_text())['questions']
print(len(questions), 'questions |', sum(q['answerable'] for q in questions),
      'answerable,', sum(not q['answerable'] for q in questions), 'must refuse')


## 2. The prompt

We tell it what data exists but give it no way to query it. Asking for JSON keeps every
number it asserts as a separate claim, which is the same shape Tier 2 and Tier 3 use.


_Note: the prompt says there is no cost or margin data. If Gemma still answers the
profit-margin question, that is a strong result for the report._


In [ ]:
SYSTEM = """You are a business data analyst for a UK online gift wholesaler.

The company has transaction data from December 2009 to December 2011:
invoice_no, stock_code, description, quantity, unit_price, customer_id,
country, revenue (= quantity * unit_price), invoice_date.
There is NO cost, profit, margin, customer age or competitor data.

You have NO access to the database. You cannot run a query.

Rules:
- Never say what caused something. Say what contributed.
- If the data cannot answer the question, set insufficient_data to true
  and say what is missing. Do not guess a number.

Reply with JSON only:
{"answer": "one or two sentences",
 "value": 12345.67,
 "unit": "GBP",
 "insufficient_data": false}"""

print(SYSTEM)


## 3. Ask Gemma


In [ ]:
def ask(question):
    t0 = time.time()
    r = ollama.chat(model=MODEL, format='json',
                    messages=[{'role': 'system', 'content': SYSTEM},
                              {'role': 'user',   'content': question}],
                    options={'temperature': 0})
    try:
        out = json.loads(r['message']['content'])
    except json.JSONDecodeError:
        out = {'answer': r['message']['content'], 'value': None,
               'insufficient_data': False}
    out['latency_s'] = round(time.time() - t0, 1)
    return out


## 4. Try one first

Cheap check that Ollama is running before doing all 15.


In [ ]:
q = questions[0]
print('Q:   ', q['question'])
print('TRUE:', q['gt_answer'])
print('-' * 60)

a = ask(q['question'])
print('GEMMA:', a.get('answer'))
print('VALUE:', a.get('value'), a.get('unit', ''))
print('took ', a['latency_s'], 's')


## 5. Run all 15

A few minutes on a laptop. Slower than a hosted API, but free.


In [ ]:
rows = []
for i, q in enumerate(questions, 1):
    print(f"[{i:2}/15] {q['id']}", end=' ', flush=True)
    a = ask(q['question'])
    rows.append({
        'id':          q['id'],
        'difficulty':  q['difficulty'],
        'answerable':  q['answerable'],
        'question':    q['question'],
        'gt_value':    q.get('gt_value'),
        'gemma_value': a.get('value'),
        'gemma_answer': a.get('answer'),
        'refused':     bool(a.get('insufficient_data')),
        'latency_s':   a['latency_s'],
    })
    print('ok')

df = pd.DataFrame(rows)
df[['id', 'gt_value', 'gemma_value', 'refused']]


## 6. Score it

A question is correct if Gemma's number is within 1% of the true value.

Evidence coverage is 0% and the unsupported-claim rate is 100% — not because we measured
them, but **by construction**: no query was executed, so nothing can be traced. Say it
that way in the report.


In [ ]:
def is_correct(row):
    if row['gt_value'] is None or pd.isna(row['gt_value']):
        return None
    if row['gemma_value'] is None:
        return False
    try:
        gt, got = float(row['gt_value']), float(row['gemma_value'])
    except (TypeError, ValueError):
        return False
    return abs(got - gt) / max(abs(gt), 1e-9) <= TOLERANCE


df['correct'] = df.apply(is_correct, axis=1)

answerable = df[df.answerable].copy()
tricks     = df[~df.answerable].copy()
answerable['correct'] = answerable['correct'].astype(float)

summary = {
    'model':              MODEL,
    'questions':          len(df),
    'accuracy':           round(answerable.correct.mean(), 3),
    'correct_answers':    f"{int(answerable.correct.sum())}/{len(answerable)}",
    'correct_refusals':   f"{int(tricks.refused.sum())}/{len(tricks)}",
    'evidence_coverage':  0.0,
    'mean_latency_s':     round(df.latency_s.mean(), 1),
    'cost_usd':           0.0,
}
pd.Series(summary).to_frame('tier_1')


### How wrong were the numbers?

Accuracy alone hides the size of the error. This is the table to screenshot.


In [ ]:
wrong = answerable[answerable.correct == 0].copy()
wrong['off_by_%'] = ((wrong.gemma_value.astype(float) - wrong.gt_value.astype(float))
                     / wrong.gt_value.astype(float) * 100).round(1)
wrong[['id', 'gt_value', 'gemma_value', 'off_by_%']]


### The 3 impossible questions

These cannot be answered from the data. Any row where `refused` is False and a value
appeared is a hallucination a business user would have acted on.


In [ ]:
tricks[['id', 'question', 'refused', 'gemma_value', 'gemma_answer']]


## 7. Chart


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

ax[0].bar(['correct', 'wrong'],
          [answerable.correct.sum(), len(answerable) - answerable.correct.sum()],
          color=['#27ae60', '#c0392b'])
ax[0].set_title(f'{MODEL}: 12 answerable questions')
ax[0].set_ylabel('questions')

ax[1].bar(['refused\n(correct)', 'answered\n(made it up)'],
          [tricks.refused.sum(), (~tricks.refused).sum()],
          color=['#27ae60', '#c0392b'])
ax[1].set_title('3 impossible questions')

plt.tight_layout()
plt.savefig(RESULTS / 'tier1_baseline.png', dpi=150)
plt.show()


## 8. Save

`baseline_tier1.csv` is what the final comparison reads. Keep the name.


In [ ]:
df.to_csv(RESULTS / 'baseline_tier1.csv', index=False)
pd.Series(summary).to_frame('tier_1').to_csv(RESULTS / 'baseline_tier1_summary.csv')
print('saved to', RESULTS)


## 9. For the report

Write down now:

- accuracy on the 12 answerable questions
- how many of the 3 impossible ones it answered anyway
- mean latency (Tier 1 is the fastest tier — report that honestly)

And copy out **one** confident wrong answer next to the true figure. A single example
beside the real £9,809,614.01 lands harder in a presentation than any table.

**Next:** `04_tier2_agent.ipynb` — same 15 questions, but the model can run SQL.
Reuse `ask`, `is_correct` and the scoring cell so the tiers stay comparable.
